# DANH GIA NGOAI TAI (downstream) - Nhom 08 - Hyena cho tieng Viet

Bo sung cho danh gia NOI TAI (perplexity, E1). Thay yeu cau phai so sanh ca
ngoai tai lan noi tai; notebook nay lo phan ngoai tai.

**Notebook chi clone repo va goi module.** Khong co logic thi nghiem nao viet
trong o code - code trong `.ipynb` khong co test nao chay qua, va bon lan chay
Kaggle hong truoc day deu vi the. Toan bo logic nam trong
`hyena_study/downstream.py`, duoc phu boi `tests/test_downstream.py` (33 test).

## Tac vu

UIT-VSFC (Vietnamese Students' Feedback Corpus), ban do chinh nhom UIT-NLP dang:
`uitnlp/vietnamese_students_feedback`. Hai tac vu phan loai cau tren CUNG mot tap cau:

| Tac vu | So lop | Nhan |
|---|---|---|
| sentiment | 3 | negative / neutral / positive |
| topic | 4 | lecturer / program / facility / others |

Chia tap: train 11.426 - validation 1.583 - test 3.166 (da doi chieu voi ban
`tridm/UIT-VSFC`, khop tuyet doi o ca ba split va moi phan bo nhan).

Trich dan goc: Nguyen et al., *UIT-VSFC: Vietnamese Students' Feedback Corpus
for Sentiment Analysis*, KSE 2018, tr. 19-24.
Giay phep tren HuggingFace ghi **unknown** - dung cho hoc tap/nghien cuu.

## Truoc khi chay

1. Settings -> Accelerator -> **GPU T4** (hoac P100).
2. Settings -> **Internet: ON** (can cho `git clone` va tai VSFC).
3. Thoi gian uoc tinh: **~20 phut huan luyen + ~2,5 gio danh gia**.

## 1. Clone repo va kiem tra moi truong

In [ ]:
import os, sys, subprocess, json, time

WORK = '/kaggle/working'
REPO_DIR = os.path.join(WORK, 'Hyena-Attention-Study')
REPO = 'https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('thu muc lam viec:', os.getcwd())

import torch, pyarrow
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('pyarrow', pyarrow.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('CANH BAO: khong thay GPU. Bat Accelerator roi chay lai.')

## 2. Chay test truoc khi dot GPU

Test day du cua module danh gia ngoai tai. **Test do thi moi chay tiep.**
Nhom test D3 la quan trong nhat: no kiem chung bang so rang bo loc Hyena phu
thuoc do dai chuoi L, nen moi chuoi phai duoc dem ve cung mot do dai co dinh.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_downstream.py', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise SystemExit('TEST THAT BAI - dung lai, khong chay thi nghiem.')

## 3. Chuan bi checkpoint

Neu da dinh kem checkpoint qua Kaggle Dataset thi dat duong dan vao `CKPT_DIR`.
Neu khong, o duoi se huan luyen lai bang **dung cau hinh E1** (`vi/syllable`,
uniform, 50 trieu token, seed 0). Ban tai hien da duoc kiem chung: PPL Hyena
trung E1 seed 0 den chu so cuoi cung.

In [ ]:
CKPT_DIR = '/kaggle/working/results'
CKPT_H = os.path.join(CKPT_DIR, 'DEMO_vi_HHHH_s0.pt')
CKPT_A = os.path.join(CKPT_DIR, 'DEMO_vi_AAAA_s0.pt')

COMMON = [
    '--lang', 'vi', '--tokenizer', 'syllable', '--vocab_size', '16000',
    '--n_docs', '90000', '--seq_len', '512', '--batch_size', '16',
    '--lr', '3e-4', '--token_budget', '50000000',
    '--data_seed', '0', '--seed', '0', '--decay_mode', 'uniform',
    '--token_cache', '/kaggle/working/data_cache',
    '--out_dir', CKPT_DIR, '--save_ckpt',
]

def run_train(layers, run_name):
    cmd = [sys.executable, '-m', 'hyena_study.train', '--layers', layers,
           '--run_name', run_name] + COMMON
    print('>>', ' '.join(cmd[2:]))
    t0 = time.time()
    r = subprocess.run(cmd, text=True)
    if r.returncode != 0:
        raise SystemExit(f'train {run_name} that bai')
    print(f'{run_name}: {time.time() - t0:.0f}s')

for path, layers, name in [(CKPT_H, 'HHHH', 'DEMO_vi_HHHH_s0'),
                           (CKPT_A, 'AAAA', 'DEMO_vi_AAAA_s0')]:
    if os.path.exists(path):
        print('da co', path)
    else:
        run_train(layers, name)

## 4. Danh gia ngoai tai

Module tu chay **bon nhanh** cho moi tac vu:

| Nhanh | Y nghia |
|---|---|
| Hyena-pretrained | Hyena da tien huan luyen |
| Transformer-pretrained | Transformer da tien huan luyen |
| Hyena-random | CUNG kien truc, trong so ngau nhien |
| Transformer-random | CUNG kien truc, trong so ngau nhien |

Hai nhanh `random` la **doi chung bat buoc**: thieu chung thi khong biet diem so
dat duoc la nho tien huan luyen hay chi nho kien truc cong dau phan loai. Module
con tinh them duong co so **doan lop dong nhat** (majority).

Chi so chinh la **macro-F1**, khong phai accuracy: lop `neutral` chi chiem ~4%
nen doan bua lop dong da co accuracy cao gia tao.

`probe` = dong bang than mo hinh, chi hoc dau tuyen tinh (do truc tiep chat luong
bieu dien). `finetune` = huan luyen toan bo. Ba seed cho moi o.

In [ ]:
def run_downstream(tag, extra):
    out = f'/kaggle/working/results/downstream_{tag}.json'
    cmd = [sys.executable, '-m', 'hyena_study.downstream',
           '--ckpt', CKPT_H, '--compare', CKPT_A,
           '--out', out, '--data_dir', '/kaggle/working/data_cache/vsfc'] + extra
    print('>>', ' '.join(cmd[2:]), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, text=True)
    if r.returncode != 0:
        raise SystemExit(f'downstream {tag} that bai')
    print(f'{tag}: {time.time() - t0:.0f}s -> {out}')
    return out

# Probe chay nhanh (dac trung tinh mot lan roi dung lai), lam truoc de co ket qua som.
p_probe = run_downstream('probe', ['--mode', 'probe', '--task', 'both',
                                   '--seeds', '0', '1', '2', '--epochs', '20'])

### 4b. Tinh chinh toan phan

Buoc nay ton thoi gian nhat (~2,5 gio). Moi chuoi duoc dem ve **L = 512**, dung
bang do dai luc tien huan luyen, vi bo loc Hyena tham so hoa tren truc thoi gian
da chuan hoa `t = linspace(0, 1, L)` - doi L la co gian ca bo loc.

In [ ]:
p_ft = run_downstream('finetune', ['--mode', 'finetune', '--task', 'both',
                                   '--seeds', '0', '1', '2', '--epochs', '5'])

### 4c. Anh huong cua do dai dem (tuy chon, re)

Chay lai **probe** voi `--pad_to 176` (vua du chua cau dai nhat, 161 token) de do
xem viec lech do dai so voi luc tien huan luyen anh huong Hyena den dau. Day la
bang chung truc tiep cho nhan xet ve phu thuoc L, do tren tac vu that.

In [ ]:
p_len = run_downstream('probe_pad176', ['--mode', 'probe', '--task', 'both',
                                        '--seeds', '0', '1', '2',
                                        '--epochs', '20', '--pad_to', '176'])

## 5. Tom tat ket qua

In [ ]:
import numpy as np
from collections import defaultdict

def summary_table(path):
    d = json.load(open(path, encoding='utf-8'))
    by = defaultdict(list)
    for r in d['runs']:
        by[(r['task'], r['mode'], r['pooling'], r['arm'])].append(r)
    print(f"\n=== {os.path.basename(path)} | pad_to={d.get('pad_to')} ===")
    print(f"{'tac vu':10s} {'che do':9s} {'nhanh':24s} {'macro-F1':>18s} {'accuracy':>18s}")
    for k in sorted(by):
        rs = by[k]
        f1 = np.array([r['test_macro_f1'] for r in rs])
        ac = np.array([r['test_accuracy'] for r in rs])
        t = 4.303 if len(rs) == 3 else float('nan')
        ci_f = t * f1.std(ddof=1) / np.sqrt(len(f1)) if len(rs) > 1 else 0.0
        ci_a = t * ac.std(ddof=1) / np.sqrt(len(ac)) if len(rs) > 1 else 0.0
        print(f"{k[0]:10s} {k[1]:9s} {k[3]:24s} "
              f"{f1.mean():.4f} +/- {ci_f:.4f}  {ac.mean():.4f} +/- {ci_a:.4f}")
    maj = {(r['task'],): (r['majority_test_macro_f1'], r['majority_test_accuracy'])
           for r in d['runs']}
    for (task,), (mf, ma) in sorted(maj.items()):
        print(f"  [majority] {task}: macro-F1 {mf:.4f} | accuracy {ma:.4f}")
    for c in d['comparisons']:
        print(f"  [so sanh] {c['task']}/{c['mode']}/{c['pooling']} {c['metric']}: {c['verdict']}")

for p in [p_probe, p_ft, p_len]:
    summary_table(p)

## 6. Dong goi ket qua

Tai `downstream_results.zip` ve roi giai nen vao `results/` cua repo.
Cac tep `.json` la **bang chung** - `.gitignore` giu lai `results/*.json`.

In [ ]:
import zipfile, glob
zpath = '/kaggle/working/downstream_results.zip'
files = sorted(glob.glob('/kaggle/working/results/downstream_*.json'))
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.basename(f))
        print('them', os.path.basename(f), os.path.getsize(f), 'bytes')
print('\nxong:', zpath, os.path.getsize(zpath), 'bytes')